<a href="https://colab.research.google.com/github/Clamata/synthetic-data/blob/main/synthetic_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Thu Nov 19 20:52:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!git clone https://github.com/patrykbart/synthetic-data-course-template.git /content/project
%cd /content/project
!pip install -e .
!synth-data

Cloning into '/content/project'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 93 (delta 43), reused 84 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (93/93), 101.50 KiB | 7.81 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/project
Obtaining file:///content/project
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 28.8 MB/s  0:00:00
  Building editable for synthetic-data-course (pyproject.toml) ... done
  Created wheel for synthetic-data-course: filename=synthetic_data_course-0.1.0-py3-none-any.whl size=6987 sha256=5907feaf4ba090a183947ec64e1d8e77f3d799e5743ba1daa5f42286c828fcf5
  Stored in directory: /tmp/

In [ ]:
# Write src/synthdata/balance.py
balance_code = '''"""Meeting 1: the two classical ways to add Egyptian Mau training examples."""

import random
from PIL import Image
from torchvision import transforms


def oversample(
    minority_photos: list[str], budget: int, rng: random.Random
) -> list[str]:
    """Choose which Egyptian Mau training photos to repeat."""
    if not minority_photos or budget <= 0:
        return []
    q, r = divmod(budget, len(minority_photos))
    chosen = list(minority_photos) * q
    if r > 0:
        chosen.extend(rng.sample(minority_photos, r))
    rng.shuffle(chosen)
    return chosen


AUGMENT_TRANSFORMS = transforms.Compose(
    [
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    ]
)


def augment(image: Image.Image) -> Image.Image:
    """Return a randomly transformed copy of one Egyptian Mau training photo."""
    return AUGMENT_TRANSFORMS(image)
'''
with open("src/synthdata/balance.py", "w") as f:
    f.write(balance_code)
print("Written default balance.py")

# Run 3 baselines (5 seeds each)
!synth-train --name m1-real --method real
!synth-train --name m1-oversample --method oversample
!synth-train --name m1-augment --method augment

# Run variant with stronger augmentations
balance_strong = '''import random
from PIL import Image
from torchvision import transforms

def oversample(minority_photos: list[str], budget: int, rng: random.Random) -> list[str]:
    if not minority_photos or budget <= 0:
        return []
    q, r = divmod(budget, len(minority_photos))
    chosen = list(minority_photos) * q
    if r > 0:
        chosen.extend(rng.sample(minority_photos, r))
    rng.shuffle(chosen)
    return chosen

AUGMENT_TRANSFORMS = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=35),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.6),
    transforms.RandomResizedCrop(256, scale=(0.7, 1.0)),
])

def augment(image: Image.Image) -> Image.Image:
    return AUGMENT_TRANSFORMS(image)
'''
with open("src/synthdata/balance.py", "w") as f:
    f.write(balance_strong)

!synth-train --name m1-augment-strong --method augment --tag variant=strong

# Restore default balance.py
with open("src/synthdata/balance.py", "w") as f:
    f.write(balance_code)

# Run synth-test on baselines
!synth-test --run m1-real
!synth-test --run m1-oversample
!synth-test --run m1-augment
!synth-test --run m1-augment-strong

# Summary report
import json, glob
print("\n" + "="*50)
print("MEETING 1 VALIDATION RESULTS:")
for r in sorted(glob.glob("runs/m1-*/results.json")):
    with open(r) as f:
        d = json.load(f)
    recalls = [s["egyptian_mau_recall"] for s in d["seeds"]]
    f1s = [s["macro_f1"] for s in d["seeds"]]
    print(f"{d['name']:20s} | Val Recall: {sum(recalls)/len(recalls):.3f} | Macro-F1: {sum(f1s)/len(f1s):.3f}")

print("\nMEETING 1 TEST RESULTS:")
for r in sorted(glob.glob("runs/m1-*/test-results.json")):
    with open(r) as f:
        d = json.load(f)
    recalls = [s["egyptian_mau_recall"] for s in d["seeds"]]
    f1s = [s["macro_f1"] for s in d["seeds"]]
    print(f"{d['name']:20s} | Test Recall: {sum(recalls)/len(recalls):.3f} | Macro-F1: {sum(f1s)/len(f1s):.3f}")
print("="*50)

Written default balance.py
m1-real: real, efficientnet_b0, 5 seeds, cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100% 20.5M/20.5M [00:00<00:00, 171MB/s]
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this Data

In [ ]:
import base64
b64 = """..."""
with open("/content/project/run_m2.py", "wb") as f:
    f.write(base64.b64decode(b64))

!python /content/project/run_m2.py

Working directory: /content/project

STARTING MEETING 2 DIFFUSION EXPERIMENTS

--- 1. Generating & Training m2-base ---

>>> Running: synth-generate --name m2-base --prompt "a photo of an egyptian mau cat, silver spotted coat, gooseberry green eyes" --steps 20
3.6 s
  [102/110] m2-base-0101 ok 3.6 s
  [103/110] m2-base-0102 ok 3.6 s
  [104/110] m2-base-0103 ok 3.6 s
  [105/110] m2-base-0104 ok 3.6 s
  [106/110] m2-base-0105 ok 3.6 s
  [107/110] m2-base-0106 ok 3.6 s
  [108/110] m2-base-0107 ok 3.6 s
  [109/110] m2-base-0108 ok 3.6 s
  [110/110] m2-base-0109 ok 3.6 s
m2-base: 110 usable images of 110; see /content/project/pools/m2-base/contact-sheet.jpg


>>> Running: synth-train --name m2-base --method synthetic --pool m2-base --tag model=sd15 --tag steps=20 --tag guidance=7.5
, efficientnet_b0, 5 seeds, cuda
  seed 0: Egyptian Mau recall 0.800, macro-F1 0.912 (33 s)
  seed 1: Egyptian Mau recall 0.700, macro-F1 0.880 (28 s)
  seed 2: Egyptian Mau recall 0.750, macro-F1 0.906 (29 s)
  

In [ ]:
!ls -lh /content/synthetic_data_full_solution.tar.gz
print("="*60)
!cat /content/project/EXPERIMENTS.md

-rw-r--r-- 1 root root 885M Nov 19 22:10 /content/synthetic_data_full_solution.tar.gz
# Experiment Log

## Meeting 1 — Real-data baselines

- **Question:** How much do oversampling and ordinary augmentation help Egyptian Mau recall compared with the imbalanced data?
- **How `oversample()` works:** Exact copies of the 15 Egyptian Mau training photos are repeated as evenly as possible using `divmod(budget, len(minority_photos))` and `rng.sample()` to draw the remainder. Every photo appears 3 or 4 times, ensuring balanced frequency and full reproducibility from the random seed.
- **How `augment()` works, and why these transforms:** `augment()` applies label-preserving geometric and photometric perturbations: `RandomHorizontalFlip(p=0.5)`, `RandomRotation(degrees=15)`, `ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)`, and mild `RandomAffine`. These transforms preserve the distinctive spotted coat pattern and feline anatomy while ensuring each epoch sees diverse variations. A sec